# ✝️ Godly Geeks Movement: Build an AI Fellowship Assistant
**CoCIS Christian Fellowship · Makerere University · 7 Nov 2026**
Facilitator: **Yiga Gilbert**, Software Engineer, Sunbird AI

<a href="https://colab.research.google.com/github/yigagilbert/Future_of_ai_session/blob/main/notebooks/workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this session we build **one project** in four steps. Each step adds one idea:

| Step | Topic | What we add |
|---|---|---|
| 1 | **LLM development** | Talk to Gemini through its API |
| 2 | **API integration** | Give it tools: a Bible API and Sunbird AI translation |
| 3 | **MCP servers** | Package the tools so *any* AI app can use them |
| 4 | **Agentic AI** | Give it a goal and let it plan and use the tools itself |

> *"I have filled him with the Spirit of God, with wisdom, with understanding, with knowledge, and with all kinds of skills."* (Exodus 31:3)

Full code to take home: **https://github.com/yigagilbert/Future_of_ai_session**

## 0. Setup (2 minutes)

1. Get a **free Gemini API key**: https://aistudio.google.com/apikey
2. In Colab, click the **🔑 key icon** in the left sidebar → *Add new secret* → name it `GEMINI_API_KEY`, paste the key, and turn on *Notebook access*.
3. *(Optional)* Add `SUNBIRD_API_KEY` the same way for Luganda translation (https://api.sunbird.ai).

If you skip the secrets, the cell below will ask you to paste the key instead.

In [ ]:
!pip install -q google-genai "mcp>=2.3,<3" requests

In [ ]:
import os
from getpass import getpass

def load_key(name, required=True):
    try:
        from google.colab import userdata
        os.environ[name] = userdata.get(name)
    except Exception:
        if required or input(f"Do you have a {name}? (y/n) ").lower().startswith("y"):
            os.environ[name] = getpass(f"Paste your {name}: ")
    print(f"{name}: {'✅ set' if os.environ.get(name) else '⚠️ not set'}")

load_key("GEMINI_API_KEY")
load_key("SUNBIRD_API_KEY", required=False)

from google import genai
from google.genai import types

MODEL = "gemini-2.5-flash"  # free tier. Change here if Google renames models.

client = genai.Client(
    # The free tier allows only a few requests per minute: retry instead of crashing.
    http_options=types.HttpOptions(retry_options=types.HttpRetryOptions(attempts=5, initial_delay=2, max_delay=30))
)
print("Ready ✅")

---
# Step 1: LLM development 🧠
An LLM is a function: **text in → text out**. We reach it through an **API** with an **API key**.

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="In two sentences, what is a large language model?",
)
print(response.text)

### System instructions and temperature
The **system instruction** sets the model's role and style. **Temperature** controls creativity (0 = predictable, higher = more varied).

In [ ]:
config = types.GenerateContentConfig(
    system_instruction=(
        "You are a friendly assistant for a Christian student fellowship at Makerere University. "
        "Explain things simply, like a senior student talking to a first-year. Keep answers under 100 words."
    ),
    temperature=0.7,
)
response = client.models.generate_content(model=MODEL, contents="Explain the meaning of John 3:16.", config=config)
print(response.text)

### Tokens: what you pay for (or what your rate limit counts)

In [ ]:
u = response.usage_metadata
print(f"prompt tokens: {u.prompt_token_count} | answer tokens: {u.candidates_token_count} | total: {u.total_token_count}")

### Chat = re-sending the history
The model has **no memory**. A chat works because the SDK sends the whole conversation every time.

In [ ]:
chat = client.chats.create(model=MODEL, config=config)
for q in ["Give me one verse about courage.", "Why did you choose that one?"]:
    print("You:", q)
    print("AI: ", chat.send_message(q).text, "\n")

### ⚠️ The limitation
Ask for live data or a specialised skill, and the model can only **guess**:

In [ ]:
print(client.models.generate_content(
    model=MODEL,
    contents="What is today's random verse of the day from bible-api.com? And translate 'God is love' into Luganda.",
).text)

👉 It has no internet access and no translation engine. **Let's give it tools.**

---
# Step 2: API integration (tool / function calling) 🔧
The model **cannot call APIs itself**. Instead:
1. We **describe** our tools (name + description + JSON schema).
2. The model replies: *"please call `get_bible_verse` with `reference='Psalm 23:1'`"*.
3. **Our code** calls the API and sends the result back.
4. The model writes the final answer with real data.

First, our tools. These are plain Python functions that call real APIs. We save them to a file so the MCP server in Step 3 can reuse them.

In [ ]:
%%writefile fellowship_tools.py
import os, re, requests
from datetime import date
from pathlib import Path

def get_bible_verse(reference: str, translation: str = "web") -> str:
    """Look up a Bible passage, e.g. "John 3:16" or "Psalm 23". translation: "web" or "kjv"."""
    try:
        r = requests.get(f"https://bible-api.com/{requests.utils.quote(reference)}",
                         params={"translation": translation}, timeout=30)
        if r.status_code == 404:
            return f"Error: could not find '{reference}'."
        r.raise_for_status()
        d = r.json()
        return f"{d['reference']} ({translation.upper()}): {' '.join(d['text'].split())}"
    except requests.RequestException as e:
        return f"Error: Bible API unreachable ({e})"

def get_random_verse() -> str:
    """Get one random Bible verse (a 'verse of the day')."""
    try:
        v = requests.get("https://bible-api.com/data/web/random", timeout=30).json()["random_verse"]
        return f"{v['book']} {v['chapter']}:{v['verse']} (WEB): {' '.join(v['text'].split())}"
    except requests.RequestException as e:
        return f"Error: Bible API unreachable ({e})"

def translate_text(text: str, target_language: str, source_language: str = "eng") -> str:
    """Translate text with Sunbird AI. ISO 639-3 codes: lug=Luganda, ach=Acholi, teo=Ateso,
    lgg=Lugbara, nyn=Runyankole, swa=Swahili, eng=English."""
    token = os.getenv("SUNBIRD_API_KEY")
    if not token:
        return "Error: SUNBIRD_API_KEY not set, so translation is unavailable."
    r = requests.post("https://api.sunbird.ai/tasks/translate",
                      headers={"Authorization": f"Bearer {token}"},
                      json={"source_language": source_language, "target_language": target_language, "text": text},
                      timeout=120)
    if r.status_code >= 400:
        return f"Error from Sunbird ({r.status_code}): {r.text[:200]}"
    return r.json()["output"]["translated_text"]

def save_devotional(title: str, content: str) -> str:
    """Save a finished devotional as a Markdown file."""
    Path("output").mkdir(exist_ok=True)
    slug = re.sub(r"[^a-z0-9]+", "-", title.lower()).strip("-")[:50] or "devotional"
    path = Path("output") / f"{date.today()}-{slug}.md"
    path.write_text(f"# {title}\n\n{content}\n", encoding="utf-8")
    return f"Saved to {path}"

ALL_TOOLS = {f.__name__: f for f in (get_bible_verse, get_random_verse, translate_text, save_devotional)}

In [ ]:
import importlib, fellowship_tools
importlib.reload(fellowship_tools)
from fellowship_tools import *

# Plain Python, no AI involved yet:
print(get_bible_verse("Psalm 23:1"))
print(translate_text("God is love", "lug"))

### Describe the tools to Gemini
This JSON schema is the **contract** between our code and the model. Remember it: MCP uses the same idea.

In [ ]:
tool_declarations = [
    types.FunctionDeclaration(
        name="get_bible_verse",
        description="Look up a Bible passage by reference, e.g. 'John 3:16' or 'Psalm 23'.",
        parameters_json_schema={
            "type": "object",
            "properties": {
                "reference": {"type": "string", "description": "Bible reference"},
                "translation": {"type": "string", "enum": ["web", "kjv"]},
            },
            "required": ["reference"],
        },
    ),
    types.FunctionDeclaration(
        name="translate_text",
        description="Translate text into a Ugandan/African language using Sunbird AI.",
        parameters_json_schema={
            "type": "object",
            "properties": {
                "text": {"type": "string"},
                "target_language": {"type": "string", "description": "ISO 639-3 code, e.g. lug=Luganda"},
            },
            "required": ["text", "target_language"],
        },
    ),
]
tool_config = types.GenerateContentConfig(
    tools=[types.Tool(function_declarations=tool_declarations)],
    system_instruction="You help a student fellowship. Always use tools for Bible text and translations; never guess.",
)

# Watch what the model sends back: a REQUEST to call a function, not an answer
r = client.models.generate_content(model=MODEL, contents="What does Psalm 23:1 say?", config=tool_config)
print(r.function_calls)

### The tool-calling loop

In [ ]:
def ask(question):
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]
    while True:
        response = client.models.generate_content(model=MODEL, contents=contents, config=tool_config)
        if not response.function_calls:            # no tool requested -> final answer
            return response.text
        contents.append(response.candidates[0].content)   # keep the model's request in history
        parts = []
        for call in response.function_calls:
            print(f"   🔧 {call.name}({dict(call.args)})")
            result = ALL_TOOLS[call.name](**call.args)  # WE run the tool
            print(f"   ↳ {result[:100]}")
            parts.append(types.Part.from_function_response(name=call.name, response={"result": result}))
        contents.append(types.Content(role="user", parts=parts))

print(ask("What does Psalm 23:1 say? Explain it in one sentence."))

In [ ]:
print(ask("Translate 'God is love' into Luganda."))

### Shortcut: automatic function calling
The SDK can run that loop for you. Pass plain Python functions; it reads their **type hints and docstrings** to build the schema.

In [ ]:
r = client.models.generate_content(
    model=MODEL,
    contents="Quote Romans 8:28 in KJV and give me a random verse too.",
    config=types.GenerateContentConfig(tools=[get_bible_verse, get_random_verse]),
)
print(r.text)

---
# Step 3: MCP servers 🔌
**Problem:** our tools are locked inside *this notebook*, wired to *one* model. What if Claude Desktop, VS Code, Cursor or another app needs them?

**MCP (Model Context Protocol)** is an open standard for exposing tools to *any* AI app. Think of it as **USB-C for AI tools**.

```
 Gemini / Claude / any LLM ◀──▶ MCP client ◀──MCP──▶ MCP server ──▶ bible-api.com, Sunbird AI
```

### The server: about 20 lines wrapping the same functions

In [ ]:
%%writefile mcp_server.py
from mcp.server.mcpserver import MCPServer
import fellowship_tools as t

mcp = MCPServer(name="fellowship-assistant")

@mcp.tool()
def get_bible_verse(reference: str, translation: str = "web") -> str:
    """Look up a Bible passage, e.g. "John 3:16". translation: "web" or "kjv"."""
    return t.get_bible_verse(reference, translation)

@mcp.tool()
def get_random_verse() -> str:
    """Get one random Bible verse (a 'verse of the day')."""
    return t.get_random_verse()

@mcp.tool()
def translate_text(text: str, target_language: str, source_language: str = "eng") -> str:
    """Translate with Sunbird AI. ISO 639-3 codes: lug=Luganda, ach=Acholi, nyn=Runyankole, swa=Swahili."""
    return t.translate_text(text, target_language, source_language)

@mcp.tool()
def save_devotional(title: str, content: str) -> str:
    """Save a finished devotional (Markdown) to a file."""
    return t.save_devotional(title, content)

if __name__ == "__main__":
    mcp.run()   # talks MCP over stdin/stdout

### The client: discover the tools, then hand them to Gemini
Note: **this client contains no tool code at all.** It asks the server what it can do.

In [ ]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER = StdioServerParameters(command=sys.executable, args=["mcp_server.py"], env=dict(os.environ))
ERRLOG = open("mcp_server.log", "w")   # Colab's stderr can't be handed to a subprocess

async with stdio_client(SERVER, errlog=ERRLOG) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        listed = await session.list_tools()
        for tool in listed.tools:
            print(f"🔌 {tool.name}\n   schema: {tool.input_schema}\n")

In [ ]:
def mcp_tools_to_gemini(mcp_tools):
    # MCP tools already carry a JSON schema, so conversion is one line each
    return types.Tool(function_declarations=[
        types.FunctionDeclaration(name=t.name, description=t.description or "", parameters_json_schema=t.input_schema)
        for t in mcp_tools
    ])

async def call_mcp_tool(session, name, args):
    result = await session.call_tool(name, args)
    return "\n".join(c.text for c in result.content if getattr(c, "text", None))

async def ask_via_mcp(question):
    async with stdio_client(SERVER, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            config = types.GenerateContentConfig(tools=[mcp_tools_to_gemini((await session.list_tools()).tools)])
            contents = [types.Content(role="user", parts=[types.Part(text=question)])]
            while True:
                response = await client.aio.models.generate_content(model=MODEL, contents=contents, config=config)
                if not response.function_calls:
                    return response.text
                contents.append(response.candidates[0].content)
                parts = []
                for call in response.function_calls:
                    print(f"   🔌 MCP → {call.name}({dict(call.args)})")
                    result = await call_mcp_tool(session, call.name, dict(call.args))
                    parts.append(types.Part.from_function_response(name=call.name, response={"result": result}))
                contents.append(types.Content(role="user", parts=parts))

print(await ask_via_mcp("Give me a verse of the day and explain it in two sentences."))

💡 **The payoff:** the *same* `mcp_server.py` can be plugged into Claude Desktop, VS Code, Cursor or Gemini CLI with a few lines of config. See the repo README.

---
# Step 4: Agentic AI 🤖
A **chatbot** answers a question. An **agent** pursues a **goal**:

```
     ┌──────────────► THINK (plan the next step)
     │                    │
  OBSERVE ◄── ACT (call a tool)
     │
     └── repeat until the goal is done (or a guardrail stops it)
```

Same loop as Step 3. What changes is the **job**: an open-ended goal that needs several tools in sequence, plus instructions that make the model plan and check its own work.

In [ ]:
MAX_STEPS = 10   # guardrail: never loop forever (and protect your free quota)

AGENT_INSTRUCTIONS = '''
You are the Fellowship Assistant, an agent that prepares material for a Christian
student fellowship at Makerere University.
1. Start by writing a short numbered PLAN.
2. Use tools for every Bible quote and every translation. Never invent scripture or translations.
3. Check each tool result. If a tool fails, adapt instead of giving up.
4. When done, save the result with save_devotional, then reply with a summary starting "DONE:".
Write for university students: warm, practical, about 250 words.
'''

async def run_agent(goal):
    async with stdio_client(SERVER, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            config = types.GenerateContentConfig(
                system_instruction=AGENT_INSTRUCTIONS,
                tools=[mcp_tools_to_gemini((await session.list_tools()).tools)],
            )
            contents = [types.Content(role="user", parts=[types.Part(text=goal)])]
            for step in range(1, MAX_STEPS + 1):
                response = await client.aio.models.generate_content(model=MODEL, contents=contents, config=config)
                contents.append(response.candidates[0].content)
                thought = "".join(p.text or "" for p in response.candidates[0].content.parts).strip()
                if thought:
                    print(f"\n🧠 [step {step}] THINK:\n{thought}")
                if not response.function_calls:
                    return thought
                parts = []
                for call in response.function_calls:
                    args = dict(call.args)
                    print(f"🔧 [step {step}] ACT: {call.name}({ {k: str(v)[:60] for k, v in args.items()} })")
                    result = await call_mcp_tool(session, call.name, args)
                    print(f"👀 [step {step}] OBSERVE: {result[:120]}")
                    parts.append(types.Part.from_function_response(name=call.name, response={"result": result}))
                contents.append(types.Content(role="user", parts=parts))
            return "Stopped: step limit reached (guardrail)."

In [ ]:
goal = (
    "Prepare this Friday's fellowship devotional on the theme 'faith over fear'. "
    "Use two Bible passages, write a short reflection and one discussion question, "
    "include a one-sentence Luganda translation of the key message, and save it."
)
final = await run_agent(goal)

In [ ]:
# Read what the agent produced
import glob
for f in sorted(glob.glob("output/*.md")):
    print("📄", f, "\n")
    print(open(f).read())

### Try your own goal

In [ ]:
await run_agent("Pick a random verse, explain it for a stressed student during exams, and give the key line in Acholi. Save it.")

---
# 🎓 Wrap-up

| Concept | One-liner |
|---|---|
| **LLM** | Text in → text out, through an API. No memory, no live data. |
| **Tool calling** | The model *asks*; your code *acts*. |
| **MCP** | A standard plug so any AI app can use your tools. |
| **Agent** | A loop: think → act → observe, until the goal is met. Add guardrails. |

### Take-home challenges
1. 🟢 Add a tool `get_hymn_of_the_day()` (a Python list of hymns is fine).
2. 🟡 Add your own API as a tool, e.g. weather for Kampala (https://open-meteo.com, no key needed).
3. 🟡 Connect `mcp_server.py` to Claude Desktop or VS Code (see the README).
4. 🔴 Build an agent that drafts the fellowship's weekly WhatsApp announcement in English **and** Luganda.

📦 **Full project:** https://github.com/yigagilbert/Future_of_ai_session

*"Whatever you do, work at it with all your heart, as working for the Lord."* (Colossians 3:23)